In [1]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [2]:
spark = (
    SparkSession.builder
    .appName("nyc-yellow-taxi-2025")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(spark.version)

4.0.3


**IMPORT FROM DRIVE**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import glob
import os

files = glob.glob("/content/drive/MyDrive/**/*.parquet", recursive=True)

for f in files[:20]:
    print(f)

/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-01.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-02.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-05.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-06.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-09.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-11.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-12.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-03.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-04.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-07.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-08.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-10.parquet


FILES CHECK

In [5]:
taxi_files = [
    f for f in files
    if "yellow_tripdata_2025" in os.path.basename(f).lower()
]

print("Yellow taxi 2025 files found:", len(taxi_files))

for f in taxi_files:
    print(f)

Yellow taxi 2025 files found: 12
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-01.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-02.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-05.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-06.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-09.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-11.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-12.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-03.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-04.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-07.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-08.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-10.parquet


**DATA SHOW**

In [15]:
trips = spark.read.parquet(*taxi_files)

trips.show(20)
trips.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-05-01 00:07:06|  2025-05-01 00:24:15|              1|          3.7|         1|                 N|         140|    

**CLEANING START**

In [7]:
cleaning_logs = []

cleaning_logs.append(("Step 1", "Loaded 12 NYC Yellow Taxi 2025 parquet files from Google Drive."))
cleaning_logs.append(("Step 2", "Checked schema and first 5 rows using printSchema() and show()."))

print("Cleaning log started")

Cleaning log started


**Count raw rows**

In [8]:
raw_count = trips.count()

print("Raw rows:", raw_count)
print("Total columns:", len(trips.columns))

cleaning_logs.append(("Step 3", f"Raw dataset has {raw_count} rows and {len(trips.columns)} columns."))

Raw rows: 48722602
Total columns: 20


**Check missing values**

In [9]:
important_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type",
    "cbd_congestion_fee"
]

missing_values = trips.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in important_cols
])

missing_values.show(truncate=False)

cleaning_logs.append(("Step 4", "Checked missing values in important columns before cleaning."))

+--------------------+---------------------+---------------+-------------+------------+------------+-----------+----------+------------+------------+------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|tip_amount|total_amount|payment_type|cbd_congestion_fee|
+--------------------+---------------------+---------------+-------------+------------+------------+-----------+----------+------------+------------+------------------+
|0                   |0                    |11611894       |0            |0           |0           |0          |0         |0           |0           |0                 |
+--------------------+---------------------+---------------+-------------+------------+------------+-----------+----------+------------+------------+------------------+



**Add useful columns**


In [10]:
df = trips

df = (
    df
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn(
        "trip_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
        )
    )
    .withColumn(
        "tip_percent",
        F.when(
            F.col("fare_amount") > 0,
            F.round((F.col("tip_amount") / F.col("fare_amount")) * 100, 2)
        ).otherwise(0)
    )
    .withColumn(
        "is_cbd",
        F.when(F.col("cbd_congestion_fee") > 0, 1).otherwise(0)
    )
)

df.select(
    "pickup_hour",
    "day_of_week",
    "pickup_month",
    "trip_minutes",
    "fare_per_mile",
    "tip_percent",
    "is_cbd"
).show(5)

cleaning_logs.append(("Step 5", "Created new columns: pickup_hour, day_of_week, pickup_month, trip_minutes, fare_per_mile, tip_percent, and is_cbd."))

+-----------+-----------+------------+------------------+-------------+-----------+------+
|pickup_hour|day_of_week|pickup_month|      trip_minutes|fare_per_mile|tip_percent|is_cbd|
+-----------+-----------+------------+------------------+-------------+-----------+------+
|          0|          5|           5|             17.15|         4.97|      26.36|     1|
|          0|          5|           5| 6.716666666666667|         8.35|       50.0|     1|
|          0|          5|           5|              7.95|         6.37|        0.0|     1|
|          0|          5|           5|25.333333333333332|          4.3|      28.68|     1|
|          0|          5|           5| 7.633333333333334|         5.56|       15.0|     1|
+-----------+-----------+------------+------------------+-------------+-----------+------+
only showing top 5 rows


**Clean invalid rows**

In [11]:
clean = df.na.drop(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
    "total_amount"
])

cleaning_logs.append(("Step 6", "Dropped rows where important fields were missing."))

clean = clean.na.fill({
    "passenger_count": 1,
    "tip_amount": 0,
    "tolls_amount": 0,
    "extra": 0,
    "mta_tax": 0,
    "improvement_surcharge": 0,
    "congestion_surcharge": 0,
    "Airport_fee": 0,
    "cbd_congestion_fee": 0
})

cleaning_logs.append(("Step 7", "Filled missing numeric values with safe defaults."))

clean = clean.filter(
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 100) &
    (F.col("fare_amount") > 0) &
    (F.col("fare_amount") <= 1000) &
    (F.col("total_amount") > 0) &
    (F.col("total_amount") <= 1000) &
    (F.col("trip_minutes") > 0) &
    (F.col("trip_minutes") <= 180) &
    (F.col("PULocationID").between(1, 265)) &
    (F.col("DOLocationID").between(1, 265)) &
    (F.col("passenger_count").between(1, 6))
)

cleaning_logs.append(("Step 8", "Removed invalid rows: zero/negative distance, zero/negative fare, invalid trip time, invalid passenger count, and invalid location IDs."))

clean.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+-----------+------------+------------------+-------------+-----------+------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_hour|day_of_week|pickup_month|trip_minutes      |fare_per_mile|tip_percent|is_cbd|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-----

**Cleaning summary**

In [12]:
clean_count = clean.count()
removed_count = raw_count - clean_count
removed_percentage = round((removed_count / raw_count) * 100, 2)

print("Raw rows:", raw_count)
print("Clean rows:", clean_count)
print("Removed rows:", removed_count)
print("Removed percentage:", removed_percentage, "%")

cleaning_logs.append(("Step 9", f"After cleaning, {clean_count} rows remained. Removed {removed_count} rows, which is {removed_percentage}% of raw data."))

Raw rows: 48722602
Clean rows: 43902255
Removed rows: 4820347
Removed percentage: 9.89 %


**Final quality check**


In [13]:
bad_rows = clean.filter(
    (F.col("trip_distance") <= 0) |
    (F.col("fare_amount") <= 0) |
    (F.col("total_amount") <= 0) |
    (F.col("trip_minutes") <= 0)
).count()

print("Bad rows after cleaning:", bad_rows)

cleaning_logs.append(("Step 10", f"Quality check completed. Bad rows after cleaning: {bad_rows}."))

Bad rows after cleaning: 0


**Save cleaning logs**

In [14]:
cleaning_log_df = spark.createDataFrame(cleaning_logs, ["step", "description"])

cleaning_log_df.show(truncate=False)

log_output_path = "/content/drive/MyDrive/NYC taxi/outputs/cleaning_logs"

cleaning_log_df.coalesce(1).write.mode("overwrite").option("header", True).csv(log_output_path)

print("Cleaning logs saved at:")
print(log_output_path)

+-------+---------------------------------------------------------------------------------------------------------------------------------------+
|step   |description                                                                                                                            |
+-------+---------------------------------------------------------------------------------------------------------------------------------------+
|Step 1 |Loaded 12 NYC Yellow Taxi 2025 parquet files from Google Drive.                                                                        |
|Step 2 |Checked schema and first 5 rows using printSchema() and show().                                                                        |
|Step 3 |Raw dataset has 48722602 rows and 20 columns.                                                                                          |
|Step 4 |Checked missing values in important columns before cleaning.                                                       

## **Cleaning Logs:**

---



1. Loaded 12 NYC Yellow Taxi 2025 Parquet files from Google Drive.

   Why: The dataset is large, so reading directly from Drive using PySpark avoids manual upload issues.

2. Checked schema and sample rows.

   Why: To understand column names, data types, and the structure of the dataset.

3. Counted raw rows.

   Why: To compare raw data size with cleaned data size.

4. Checked missing values in important columns.

   Why: Missing values can affect analysis accuracy.

5. Created new columns: pickup_hour, day_of_week, pickup_month, trip_minutes, fare_per_mile, tip_percent, and is_cbd.

   Why: These columns are useful for time-based, fare-based, tip-based, and congestion-fee analysis.

6. Dropped rows with missing essential fields.

   Why: Trips without important fields like pickup time, location, fare, or distance cannot be used properly.

7. Filled missing numeric values with safe defaults.

   Why: To avoid removing useful rows when missing values can be handled safely.

8. Removed invalid trips with zero/negative distance, zero/negative fare, invalid duration, invalid passenger count, and invalid location IDs.

   Why: These records are not realistic and can create wrong analysis results.

9. Performed final quality check.

   Why: To confirm that no invalid rows remain after cleaning.

10. Saved cleaned data as Parquet partitioned by pickup_month.

    Why: Parquet is efficient for big data, and monthly partitioning makes future analysis faster.